# Analysis: plan quotes on a map

- Aggregates the compiled CSVs (`data/csv/`, from `compile_csvs.ipynb`) to one value per county
- Draws it on Census TIGER county boundaries in an ipyleaflet map
- Quotes are collected per county, so the county map shows the data as it was collected
- The last section is an optional alternative: the same values mapped back onto ZIP codes
- Scraped quotes are research data, not insurance advice or eligibility guidance

## Configuration

In [ ]:
from pathlib import Path

CSV_DIR = Path("data/csv")
STATES = None                # None = every state in the CSVs, or a list, e.g. ["AK", "HI"]

METRIC = "premium_monthly"   # any numeric column: premium_monthly, deductible_individual, moop_individual, ...
STAT = "median"              # median | min | max | mean, taken over the plans in each county
FILTERS = {                  # column: value or list of values; rows must match all of them
    "applicant_age": 40,
    "has_spouse": False,
    "num_dependents": 0,
    "metal_level": "SILVER",
}

TIGER_DIR = Path("data/tiger")  # Census TIGER/Line shapefiles, downloaded when first needed
TIGER_YEAR = 2025

## Load and aggregate

- Dask reads every CSV and keeps only the rows matching `FILTERS`
- What is left is small enough for pandas
- The result is one row per county

In [ ]:
import dask.dataframe as dd
import pandas as pd

quotes = dd.read_csv(str(CSV_DIR / "*" / "*.csv"), dtype={"zip_code": str, "county_fips": str, "plan_id": str}, assume_missing=True)
if STATES:
    quotes = quotes[quotes["state"].isin(STATES)]
for column, allowed in FILTERS.items():
    quotes = quotes[quotes[column].isin(allowed if isinstance(allowed, list) else [allowed])]
quotes = quotes.compute()
if quotes.empty:
    raise ValueError(f"no rows match FILTERS = {FILTERS}")

# A county quoted more than once (re-runs, or older per-ZIP runs): keep the latest quote of each scenario + plan.
SCENARIO = ["county_fips", "applicant_age", "applicant_gender", "has_spouse", "num_dependents", "household_income", "plan_id"]
quotes = quotes.sort_values("retrieved_at_utc").drop_duplicates(SCENARIO, keep="last")


def aggregate(by: list[str]) -> pd.DataFrame:
    """`STAT` of `METRIC` over the plans in each group, with the range and plan count."""
    return (
        quotes.groupby(by)
        .agg(value=(METRIC, STAT), low=(METRIC, "min"), high=(METRIC, "max"), plans=("plan_id", "nunique"), issuers=("issuer", "nunique"))
        .reset_index()
    )


by_county = aggregate(["state", "county_fips"])
states = sorted(by_county["state"].unique())
print(f"{len(quotes):,} quotes -> {len(by_county):,} counties in {states}")
by_county.groupby("state")["value"].describe().round(0)

## County boundaries

- States and counties come from the Census Bureau, downloaded once into `TIGER_DIR`
- Only the selected states are read from the national files

In [ ]:
import urllib.request
import zipfile

import geopandas as gpd

TIGER_URL = "https://www2.census.gov/geo/tiger/TIGER{year}/{folder}/tl_{year}_us_{name}.zip"
TIGER_FOLDERS = {"state": "STATE", "county": "COUNTY", "zcta520": "ZCTA520"}
SIMPLIFY_DEG = 0.001  # about 100 m; keeps the shapes sent to the browser small


def read_tiger(name: str, column: str, values) -> gpd.GeoDataFrame:
    """Shapes from a national TIGER file whose `column` is one of `values`, downloading the file if missing."""
    shp = TIGER_DIR / f"tl_{TIGER_YEAR}_us_{name}" / f"tl_{TIGER_YEAR}_us_{name}.shp"
    if not shp.exists():
        url = TIGER_URL.format(year=TIGER_YEAR, folder=TIGER_FOLDERS[name], name=name)
        print(f"downloading {url}")
        shp.parent.mkdir(parents=True, exist_ok=True)
        archive, _ = urllib.request.urlretrieve(url, shp.parent / "download.zip")
        with zipfile.ZipFile(archive) as z:
            z.extractall(shp.parent)
        Path(archive).unlink()
    where = f"{column} IN ({','.join(repr(str(v)) for v in sorted(set(values)))})"
    gdf = gpd.read_file(shp, where=where, engine="pyogrio").to_crs(epsg=4326)
    gdf["geometry"] = gdf.geometry.simplify(SIMPLIFY_DEG, preserve_topology=True)
    return gdf


state_shapes = read_tiger("state", "STUSPS", states)[["STUSPS", "NAME", "geometry"]]
county_shapes = read_tiger("county", "GEOID", by_county["county_fips"])[["GEOID", "NAMELSAD", "geometry"]]

county_map = county_shapes.merge(by_county, left_on="GEOID", right_on="county_fips").rename(columns={"NAMELSAD": "name"})
missing = sorted(set(by_county["county_fips"]) - set(county_map["county_fips"]))
print(f"{len(county_map):,} counties drawn; {len(missing)} quoted counties have no boundary: {missing}")

## County map

- Hover a county for its value, range and plan count
- The colour scale is clipped to the 2nd-98th percentile so a few extreme counties do not wash it out

In [ ]:
import json

import ipywidgets as widgets
import numpy as np
from branca.colormap import linear
from ipyleaflet import Choropleth, ColormapControl, FullScreenControl, GeoJSON, LayersControl, Map, WidgetControl

CAPTION = f"{STAT} {METRIC}"


def choropleth(shapes: gpd.GeoDataFrame, key: str, outlines: dict | None = None) -> Map:
    """Map of `shapes` coloured by their `value` column. `key` is the id column; `outlines` are named overlay shapes."""
    vmin, vmax = (float(v) for v in np.nanpercentile(shapes["value"], (2, 98)))
    vmax = max(vmax, vmin + 1)
    colormap = linear.YlOrRd_09.scale(vmin, vmax)

    geo = json.loads(shapes.to_json())
    for feature in geo["features"]:
        feature["id"] = feature["properties"][key]
    layer = Choropleth(
        geo_data=geo,
        choro_data=dict(zip(shapes[key], shapes["value"].astype(float))),
        colormap=colormap,
        value_min=vmin,
        value_max=vmax,
        style={"fillOpacity": 0.75, "weight": 0.5, "color": "#444"},
        hover_style={"fillOpacity": 0.95, "weight": 2, "color": "#000"},
        name=CAPTION,
    )
    info = widgets.HTML("<i>hover an area</i>")

    def on_hover(feature, **_):
        p = feature["properties"]
        info.value = (
            f"<b>{p['name']}</b> ({p['state']})<br>{CAPTION}: <b>{p['value']:,.0f}</b><br>"
            f"range {p['low']:,.0f} to {p['high']:,.0f}<br>{p['plans']} plans, {p['issuers']} issuers"
        )

    layer.on_hover(on_hover)

    # Centre on the first state; skip shapes that straddle the antimeridian (Aleutians).
    first = shapes[shapes["state"] == shapes["state"].iloc[0]]
    bounds = first.geometry.bounds
    minx, miny, maxx, maxy = first[(bounds["maxx"] - bounds["minx"]) < 180].total_bounds
    m = Map(center=((miny + maxy) / 2, (minx + maxx) / 2), zoom=6, scroll_wheel_zoom=True, layout=widgets.Layout(height="650px"))
    m.add(layer)
    for name, outline in (outlines or {}).items():
        m.add(GeoJSON(data=json.loads(outline.to_json()), style={"color": "#000", "weight": 1.5, "fillOpacity": 0}, name=name))
    m.add(WidgetControl(widget=info, position="topright"))
    m.add(ColormapControl(caption=CAPTION, colormap=colormap, value_min=vmin, value_max=vmax, position="bottomright"))
    m.add(LayersControl(position="topleft"))
    m.add(FullScreenControl())
    return m


choropleth(county_map, key="county_fips", outlines={"States": state_shapes})

## Tables

- The county values
- The cheapest matching plan in each county

In [ ]:
display(county_map[["state", "county_fips", "name", "value", "low", "high", "plans", "issuers"]].sort_values(["state", "value"]))

cheapest = quotes.sort_values("premium_monthly").groupby(["state", "county_fips"], as_index=False).first()
cheapest[["state", "county_fips", "issuer", "plan_name", "metal_level", "premium_monthly", "deductible_individual", "moop_individual"]].head(10)

## Alternative: map back to ZIP codes

- Optional
- Gives each ZIP the plans of the county (or counties) it is in, using the ZIP-county tables that `run_scrapers.ipynb` saved in `data/reference/zip_counties/`
- Draws them on Census ZCTA polygons
- This is derived, not collected: a ZIP is assumed to see its county's plans
- ZCTAs approximate ZIP codes, and PO-box ZIPs have no polygon
- The national ZCTA file is about 500 MB to download

In [ ]:
ZIP_COUNTIES_DIR = Path("data/reference/zip_counties")

zip_counties = pd.concat([pd.read_csv(f, dtype=str) for f in sorted(ZIP_COUNTIES_DIR.glob("*.csv"))])
county_quotes = quotes  # keep the collected, county-level quotes
quotes = quotes.drop(columns="zip_code").merge(zip_counties[["zip_code", "county_fips"]], on="county_fips")
by_zip = aggregate(["state", "zip_code"])
quotes = county_quotes

zcta_shapes = read_tiger("zcta520", "ZCTA5CE20", by_zip["zip_code"])[["ZCTA5CE20", "geometry"]]
zip_map = zcta_shapes.merge(by_zip, left_on="ZCTA5CE20", right_on="zip_code")
zip_map["name"] = zip_map["zip_code"]
print(f"{len(by_zip):,} ZIPs; {len(zip_map):,} drawn, {len(by_zip) - len(zip_map)} have no ZCTA polygon")

choropleth(zip_map, key="zip_code", outlines={"Counties": county_shapes, "States": state_shapes})